# Positional vs. lexical vs. reflexive binding: an interchange-intervention pilot

A small replication of the central distinction in Gur-Arieh et al. (2025), *Mixing Mechanisms: How Language Models Retrieve Bound Entities In-Context* ([arXiv:2510.06182](https://arxiv.org/abs/2510.06182)), run on a 1.5B instruction-tuned model with `nnsight`.

## The design

A base context lists `n` groups `E_i loves A_i` and asks `Who loves A_k?`, so the answer is `E_k`. A counterfactual context keeps the **same entities at the same positions** but permutes the attributes by `pi`, and queries the attribute now sitting at position `kp`. We copy the residual stream at the final token of the counterfactual run into the same layer and position of the base run.

Three candidate answers are then distinguishable in a single patched forward pass:

| outcome | predicted answer | interpretation |
| --- | --- | --- |
| `E_kp` | entity at the counterfactual's query *index* | positional retrieval |
| `E_pi(kp)` | entity paired in the base with the counterfactual's query *attribute* | lexical retrieval |
| `E_k` | the original base answer | the patch had no effect |

Trials are generated so that these three entities are always distinct, which is what makes the readout a three-way attribution rather than a binary one.

**A constraint worth stating up front:** this is impossible for `n = 2`. With two groups, `kp != k` forces `kp` to be the other index, and requiring `pi(kp) != kp` then forces `pi(kp) == k`, which collides with the base answer. So separating positional from lexical retrieval needs at least three groups. `test_trials.py` asserts this.

**What the design cannot resolve on its own:** a reflexive pointer to the target token predicts the same answer as the positional mechanism, because the entities sit at the same positions in both contexts. The reflexive control below replaces the entity at position `kp` in the base context with a fresh entity `F`. A positional signal then retrieves `F`, whereas a reflexive pointer to `E_kp` has no referent and should fail.

In [ ]:
!pip -q install nnsight transformers accelerate

import os
import subprocess

REPO = "https://github.com/lldliys/binding-mechanisms-pilot.git"

# Fetch binding_pilot.py when running from a bare Colab runtime. Idempotent, so
# re-running this cell after a session restart is safe.
if not os.path.exists("binding_pilot.py"):
    subprocess.run(["git", "clone", "-q", REPO], check=False)
    os.chdir("binding-mechanisms-pilot")

import random

import matplotlib.pyplot as plt
import pandas as pd
import torch

import binding_pilot as bp

os.makedirs("figures", exist_ok=True)
print("working dir:", os.getcwd())

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"  # ungated; Llama-3.2-1B-Instruct also works
BACKEND = "nnsight"
N_GROUPS = 6
N_TRIALS = 24
SEED = 0

runner = bp.Runner(MODEL, dtype="float16", prefer_nnsight=BACKEND == "nnsight")
print(f"{runner.n_layers} layers on {runner.device}, nnsight active: {runner.lm is not None}")

# Candidate-restricted decoding compares whole names, so every name must be a
# single token. Multi-token names would silently turn this into a prefix match.
entities = runner.single_token_words(bp.ENTITY_POOL)
attrs = runner.single_token_words(bp.ATTR_POOL)
print(f"single-token entities {len(entities)}/{len(bp.ENTITY_POOL)}, "
      f"attributes {len(attrs)}/{len(bp.ATTR_POOL)}")
print("dropped entities:", sorted(set(bp.ENTITY_POOL) - set(entities)))
print("dropped attributes:", sorted(set(bp.ATTR_POOL) - set(attrs)))

In [ ]:
rng = random.Random(SEED)
prefix_fn = lambda: bp.few_shot_prefix(rng, entities, attrs, N_GROUPS, n_shot=2)

trials = [bp.make_trial(rng, N_GROUPS, entities, attrs, prefix_fn) for _ in range(N_TRIALS)]

t = trials[0]
print(t.base_prompt)
print("\n--- counterfactual ---\n" + t.cf_prompt.split("\n\n")[-1])
print(f"\nunchanged={t.base_answer}  positional={t.positional}  lexical={t.lexical}")

# The intervention is only interpretable where the model solves the task cleanly.
acc = bp.clean_accuracy(runner, trials, BACKEND)
print(f"\nclean accuracy (n={N_GROUPS}): {acc:.3f}")

# The two backends implement the same intervention; they should agree to
# floating-point noise. This is the cheapest guard against a silent hook bug.
if runner.lm is not None:
    _, resid = runner.run(t.cf_prompt, save_resid=True, backend="nnsight")
    mid = runner.n_layers // 2
    a, _ = runner.run(t.base_prompt, patch=(mid, -1, resid[mid]), backend="nnsight")
    b, _ = runner.run(t.base_prompt, patch=(mid, -1, resid[mid]), backend="hooks")
    print(f"nnsight vs hooks, max|delta logits| = {(a - b).abs().max():.2e}")

In [ ]:
rows = bp.interchange_sweep(runner, trials, BACKEND)
for r in rows:
    r["positional_name"] = trials[r["trial"]].positional

df = pd.DataFrame(rows)
for name in ("positional", "lexical", "unchanged"):
    df[name] = df.label.eq(name)
by_layer = df.groupby("layer")[["positional", "lexical", "unchanged"]].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
by_layer.plot(ax=ax[0], marker="o", ms=3)
ax[0].set(xlabel="patched layer", ylabel="rate", title="Which mechanism does the patch carry?")
ax[0].axhline(1 / (N_GROUPS + 1), ls=":", c="grey", label="uniform over candidates")
ax[0].legend(fontsize=8)

piv = df.pivot_table(index="layer", columns="kp", values="positional")
im = ax[1].imshow(piv.values, aspect="auto", origin="lower", vmin=0, vmax=1, cmap="magma")
ax[1].set(xlabel="queried position kp (0-indexed)", ylabel="patched layer",
          title="Positional rate by retrieved position")
ax[1].set_xticks(range(len(piv.columns)), piv.columns)
fig.colorbar(im, ax=ax[1])
plt.tight_layout()
plt.savefig("figures/mechanism_rates.png", dpi=150)
plt.show()

peak = int(by_layer.positional.idxmax())
print(f"peak positional layer {peak}: {by_layer.positional[peak]:.3f} positional, "
      f"{by_layer.lexical[peak]:.3f} lexical, {by_layer.unchanged[peak]:.3f} unchanged")
print("\npositional rate at edge vs middle positions:")
edges = df[df.kp.isin([0, N_GROUPS - 1])].positional.mean()
middle = df[~df.kp.isin([0, N_GROUPS - 1])].positional.mean()
print(f"  edges  {edges:.3f}\n  middle {middle:.3f}")

In [ ]:
# Control 1: a norm-matched random direction. Anything this reproduces is an
# artefact of perturbing the final token, not of the information we copied.
rand = pd.DataFrame(bp.interchange_sweep(runner, trials, BACKEND, random_patch=True))
rand["positional"] = rand.label.eq("positional")

# Control 2: a permutation null for the positional rate. Shuffling which outcome
# belongs to which trial destroys the pairing while keeping the marginal
# distribution of predicted names, so it prices in name-frequency biases.
null = bp.permutation_null(rows)
null_df = pd.DataFrame(null).T.sort_index()

# Control 3: replace the entity at position kp in the base context. A positional
# signal now retrieves the fresh entity; a reflexive pointer has no referent.
swap = pd.DataFrame(bp.interchange_sweep(runner, trials, BACKEND, swapped=True))
swap["positional"] = swap.label.eq("positional")
swap["lexical"] = swap.label.eq("lexical")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(by_layer.index, by_layer.positional, marker="o", ms=3, label="observed")
ax.plot(null_df.index, null_df.null_p95, ls="--", c="grey", label="permutation null, p95")
ax.plot(rand.groupby("layer").positional.mean(), ls=":", c="crimson", label="random direction")
ax.set(xlabel="patched layer", ylabel="positional rate", title="Observed effect against controls")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("figures/controls.png", dpi=150)
plt.show()

print(f"at layer {peak}, entity at kp intact vs replaced:")
print(f"  positional  {by_layer.positional[peak]:.3f} -> "
      f"{swap[swap.layer.eq(peak)].positional.mean():.3f}")
print(f"  lexical     {by_layer.lexical[peak]:.3f} -> "
      f"{swap[swap.layer.eq(peak)].lexical.mean():.3f}")

In [ ]:
# Logit lens on the clean runs: at which depth does the answer become readable
# at the final token? This says where to expect the patch to bite.
lens = pd.DataFrame(bp.logit_lens_curve(runner, trials, BACKEND))
agg = lens.groupby("layer").agg(p_gold=("p_gold", "mean"), rank_gold=("rank_gold", "median"))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(agg.index, agg.p_gold, marker="o", ms=3)
ax[0].set(xlabel="layer", ylabel="p(correct entity)", title="Logit lens, final token")
ax[1].semilogy(agg.index, agg.rank_gold.clip(lower=1), marker="o", ms=3)
ax[1].set(xlabel="layer", ylabel="median rank of correct entity", title="Rank of the answer")
plt.tight_layout()
plt.savefig("figures/logit_lens.png", dpi=150)
plt.show()

print(agg.round(4).to_string())

## What I found

*Fill this in from the numbers printed above — write down what actually happened, including the parts that did not work.*

- Clean accuracy at `n = 6`: ...
- Peak positional layer and rate, against the permutation null and the random-direction control: ...
- Edge vs. middle positions: ...
- Reflexive control (entity at `kp` replaced): if the positional rate survives, the signal indexes a position; if it collapses, it was a pointer to the target token.

## Limitations I am aware of

- One 1.5B model, one prompt template, 24 trials per condition. The numbers are indicative, not tight.
- I patch the whole residual stream at the final token, so this localises *depth*, not the attention heads that write the signal. Attributing it to heads and asking whether they write into shared or separate subspaces is the next step, and needs DAS rather than plain patching.
- Positional and reflexive retrieval are only separated by the entity-replacement control, which also changes the token distribution of the context. A cleaner separation would keep the entity present but move it.
- Attribute permutation changes the whole context, so the counterfactual differs from the base in more than the queried group. Matched single-swap counterfactuals would be tighter.